In [ ]:
# Microsoft Foundry GPT-5.1 - Managed Identity 인증 샘플

Azure Managed Identity(관리 ID)를 사용하여 Microsoft Foundry의 GPT-5.1 모델에 인증하고 기본 프롬프트를 수행하는 샘플 코드입니다.

- **인증 방식**: `DefaultAzureCredential` (Managed Identity 우선)
- **SDK**: `openai` (Azure OpenAI 호환)
- **모델**: GPT-5.1

In [ ]:
%pip install openai azure-identity

In [ ]:
# 환경 설정
FOUNDRY_RESOURCE = "mskr-ai-foundry-resource"
ENDPOINT = f"https://{FOUNDRY_RESOURCE}.openai.azure.com/"  # Microsoft Foundry 엔드포인트
DEPLOYMENT_NAME = "gpt-5.1"  # 배포된 모델 이름
API_VERSION = "2025-04-01-preview"  # API 버전

# User-Assigned Managed Identity Client ID
MANAGED_IDENTITY_CLIENT_ID = "2ad95969-2f55-4f0c-b1c4-881219883723"

In [ ]:
from azure.identity import ManagedIdentityCredential, get_bearer_token_provider
from openai import AzureOpenAI

# User-Assigned Managed Identity를 Client ID로 명시적 지정
credential = ManagedIdentityCredential(client_id=MANAGED_IDENTITY_CLIENT_ID)
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

# Azure OpenAI 클라이언트 생성 (Managed Identity 인증)
client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version=API_VERSION,
)

print("✅ Managed Identity 인증 클라이언트 생성 완료")

In [ ]:
# 기본 Chat Completion 샘플
response = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "갤럭시 S26에 대한 시장 반응 알려줘"},
    ],
    max_completion_tokens=500,
    temperature=0.7,
)

print(response.choices[0].message.content)

In [ ]:
# 스트리밍 방식 샘플
stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Python으로 피보나치 수열을 구하는 함수를 작성해 주세요."},
    ],
    max_completion_tokens=500,
    temperature=0.7,
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)